# TPC-DS Benchmark Results Analysis
**Fabric Lakehouse SQL Endpoint vs Fabric Warehouse**

This notebook loads benchmark results from `results/` and produces:
- Summary statistics (median, p90, p95) per endpoint, query and scale factor
- Cold vs. warm cache comparison
- Lakehouse table configuration comparison (default / partitioned / zorder / vorder)
- Bar charts and box plots of query latency

In [1]:
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

ImportError: DLL load failed while importing hashing: Una directiva de Control de aplicaciones bloqueó este archivo.

## 1. Load results

In [ ]:
RESULTS_DIR = '../results'

csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'benchmark_*.csv')))
print(f'Found {len(csv_files)} result file(s):')
for f in csv_files:
    print(' ', f)

df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'\nTotal rows: {len(df):,}')
df.head()

## 2. Data quality check

In [ ]:
print('Status breakdown:')
print(df['status'].value_counts())
print()

# Filter to successful runs only
df_ok = df[df['status'] == 'success'].copy()
print(f'Successful runs: {len(df_ok):,} / {len(df):,}')
print()
print('Endpoints present:', df_ok['endpoint'].unique().tolist())
print('Scale factors:',     df_ok['scale_factor'].unique().tolist())
print('Queries:',           df_ok['query_id'].unique().tolist())

## 3. Summary statistics (median / p90 / p95)

In [ ]:
def pct(p):
    return lambda x: x.quantile(p / 100)

summary = (
    df_ok
    .groupby(['scale_factor', 'endpoint', 'query_id', 'cache_mode'])['elapsed_ms']
    .agg(
        median=pct(50),
        p90=pct(90),
        p95=pct(95),
        mean='mean',
        count='count'
    )
    .reset_index()
    .sort_values(['scale_factor', 'cache_mode', 'query_id', 'median'])
)
summary

## 4. Cold vs. warm — median latency per query

In [ ]:
ENDPOINT_ORDER = ['lakehouse_default', 'lakehouse_partitioned', 'lakehouse_vorder', 'warehouse']

for sf in sorted(df_ok['scale_factor'].unique()):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
    fig.suptitle(f'Median latency — {sf}', fontsize=14)

    for ax, cache_mode in zip(axes, ['cold', 'warm']):
        data = summary[
            (summary['scale_factor'] == sf) &
            (summary['cache_mode'] == cache_mode)
        ]
        sns.barplot(
            data=data, x='query_id', y='median',
            hue='endpoint', hue_order=ENDPOINT_ORDER, ax=ax
        )
        ax.set_title(f'{cache_mode.capitalize()} cache')
        ax.set_xlabel('Query')
        ax.set_ylabel('Median elapsed (ms)')
        ax.legend(title='Endpoint', bbox_to_anchor=(1.01, 1), loc='upper left')

    plt.tight_layout()
    plt.show()

## 5. Lakehouse config comparison (warm, all queries)

In [ ]:
lh_endpoints = [e for e in df_ok['endpoint'].unique() if e.startswith('lakehouse_')]
df_lh = df_ok[(df_ok['endpoint'].isin(lh_endpoints)) & (df_ok['cache_mode'] == 'warm')]

for sf in sorted(df_lh['scale_factor'].unique()):
    data = df_lh[df_lh['scale_factor'] == sf]
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.barplot(data=data, x='query_id', y='elapsed_ms', hue='endpoint', ax=ax,
                estimator='median', errorbar=('pi', 90))
    ax.set_title(f'Lakehouse config comparison — {sf} (warm, median ± p90 band)')
    ax.set_xlabel('Query')
    ax.set_ylabel('Elapsed (ms)')
    ax.legend(title='Config', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 6. Distribution — box plots (warm)

In [ ]:
for sf in sorted(df_ok['scale_factor'].unique()):
    data = df_ok[(df_ok['scale_factor'] == sf) & (df_ok['cache_mode'] == 'warm')]
    fig, ax = plt.subplots(figsize=(16, 6))
    sns.boxplot(data=data, x='query_id', y='elapsed_ms', hue='endpoint', ax=ax,
                flierprops=dict(marker='o', markersize=3))
    ax.set_title(f'Latency distribution — {sf} (warm)')
    ax.set_xlabel('Query')
    ax.set_ylabel('Elapsed (ms)')
    ax.legend(title='Endpoint', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 7. Lakehouse vs Warehouse — aggregate comparison

In [ ]:
df_ok['engine'] = df_ok['endpoint'].apply(
    lambda e: 'warehouse' if e == 'warehouse' else 'lakehouse'
)
engine_summary = (
    df_ok[df_ok['cache_mode'] == 'warm']
    .groupby(['scale_factor', 'engine', 'query_id'])['elapsed_ms']
    .median()
    .reset_index()
    .rename(columns={'elapsed_ms': 'median_ms'})
)

for sf in sorted(engine_summary['scale_factor'].unique()):
    data = engine_summary[engine_summary['scale_factor'] == sf]
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=data, x='query_id', y='median_ms', hue='engine', ax=ax)
    ax.set_title(f'Lakehouse (best config) vs Warehouse — {sf} — warm — median latency')
    ax.set_xlabel('Query')
    ax.set_ylabel('Median elapsed (ms)')
    plt.tight_layout()
    plt.show()